In [1]:
SEED = 42

# Data configuration
DATA_ID = 'Fancy-MLLM/R1-Onevision'
DATA_SIZE = 1250
TEST_RATIO = 0.1

In [ ]:
from datasets import load_dataset, Dataset

def load_hf_dataset(data_id, split, total_size):
    dataset_stream = load_dataset(data_id, split=split, streaming=True)

    # Slice from the start
    sliced_data = []
    for example in dataset_stream:
        if len(sliced_data) >= total_size:
            break
        sliced_data.append(example)

    dataset = Dataset.from_list(sliced_data)
    return dataset

# Load the SFT CoT dataset
dataset = load_hf_dataset(
    DATA_ID, 
    split='train',
    total_size=DATA_SIZE*9, # Load more than needed to ensure we have enough after preprocessing
)

print(dataset)

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Dataset({
    features: ['id', 'image', 'conversations'],
    num_rows: 11250
})


In [ ]:
# Convert to pandas DataFrame for easier manipulation
df = dataset.to_pandas()
df = df.rename(columns={'image': 'image_base64'})
df = df.drop_duplicates('image_base64')
df

,id,image_base64,conversations
0,fab1c569-6e43-48b8-963d-fc68d88b7923,iVBORw0KGgoAAAANSUhEUgAAAXwAAADjCAYAAACGoK/oAA...,"[{'from': 'human', 'value': '<image> Hint: Ple..."
1,b64aafe6-8b82-4e71-9c0d-f9b0c02f9dbc,iVBORw0KGgoAAAANSUhEUgAAAhkAAAEMCAYAAABp6aQ+AA...,"[{'from': 'human', 'value': '<image> Hint: Ple..."
2,8c5aa8b2-5f9a-4f61-a898-d027a8a1fe3c,iVBORw0KGgoAAAANSUhEUgAAAg8AAADxCAYAAABMIlTiAA...,"[{'from': 'human', 'value': '<image> Hint: Ple..."
3,7840e5f5-022a-46be-b0fd-9b688032dd54,iVBORw0KGgoAAAANSUhEUgAAAVkAAADMCAYAAADZLtCfAA...,"[{'from': 'human', 'value': '<image> Hint: Ple..."
4,8656a4ff-1fd0-47ed-9b36-b8131cc0de6f,iVBORw0KGgoAAAANSUhEUgAAAZsAAAEACAYAAABlOdt4AA...,"[{'from': 'human', 'value': '<image> Hint: Ple..."
...,...,...,...
11205,3f90fa06-3b7d-4824-a454-c1abcc5882aa,iVBORw0KGgoAAAANSUhEUgAAAGAAAABgCAIAAABt+uBvAA...,"[{'from': 'human', 'value': '<image> Hint: Ple..."
11210,9586239b-278d-45d2-bf07-5a4da90ddbbb,iVBORw0KGgoAAAANSUhEUgAAAJQAAABNCAIAAADPb9pkAA...,"[{'from': 'human', 'value': '<image> Hint: Ple..."
11211,12de3e65-acd9-42c1-8573-7dd052ae0b5e,iVBORw0KGgoAAAANSUhEUgAAAU4AAACPCAIAAAAa8qWbAA...,"[{'from': 'human', 'value': '<image> Hint: Ple..."
11221,0a8bc0d0-2cdc-49bf-9f54-a7fcd2d6aa5b,iVBORw0KGgoAAAANSUhEUgAAAP0AAABoCAIAAAC1/GFbAA...,"[{'from': 'human', 'value': '<image> Hint: Ple..."


In [ ]:
import pandas as pd
import re

# Extract human and assistant values from conversations
def extract_values(convo):
    human_val = None
    assistant_val = None
    for msg in convo:
        if msg.get('from') == 'human':
            human_val = msg.get('value')
        elif msg.get('from') == 'assistant':
            assistant_val = msg.get('value')
    return pd.Series([human_val, assistant_val])

df[['instruction', 'response']] = df['conversations'].apply(extract_values)

# Clean the human instruction
def clean_instruction(instruction):
    instruction = instruction.replace("<image>\nHint: ", "")          # Remove the image tag and hint
    instruction = instruction.replace("\nQuestion:", "\n\nQuestion:") # Add an extra newline before the question for better formatting
    
    # Add explicit instruction for final answer formatting
    first_line, rest = instruction.split("\n", 1)
    first_line += " Provide your final answer between <answer> and </answer>."
    instruction = first_line + "\n" + rest
    
    # Global whitespace normalization
    instruction = "\n".join(
        " ".join(line.split())
        for line in instruction.split("\n")
    )
    
    return instruction

df['instruction'] = df['instruction'].apply(clean_instruction)

# Clean the assistant response
def clean_response(response):
    response = response.replace("**Answer:**", "Answer:")

    # Extract answer from anywhere
    ans_match = re.search(r'<answer>\s*([A-Za-z])\s*</answer>', response)
    answer = ans_match.group(1) if ans_match else ""

    # Remove ALL <answer> tags from text
    response = re.sub(r'<answer>.*?</answer>', '', response, flags=re.DOTALL)

    # Optional fallback: plain "Answer: X"
    if not answer:
        m = re.search(r'(?m)^Answer:\s*([A-Za-z])', response)
        if m:
            answer = m.group(1)
            response = re.sub(r'(?m)^Answer:.*$', '', response)

    # Remove trailing blank lines inside <think>
    def clean_think_block(match):
        content = match.group(1)
        # Strip trailing whitespace/newlines inside think
        content = re.sub(r'\s+$', '', content)
        return f"<think>\n{content}\n</think>"

    response = re.sub(
        r'<think>\s*(.*?)\s*</think>',
        clean_think_block,
        response,
        flags=re.DOTALL
    )

    # Global whitespace normalization
    response = "\n".join(
        " ".join(line.split())
        for line in response.split("\n")
    ).strip()

    # Append final answer
    if answer:
        response += f"\n\n<answer>{answer}</answer>"

    return response

df['response'] = df['response'].apply(clean_response)

In [36]:
# Filter to keep only rows with valid <think> and <answer> tags
mask = (
    df['response'].str.contains("<think>", na=False) & 
    df['response'].str.contains("</think>", na=False) &
    df['response'].str.contains("<answer>", na=False) & 
    df['response'].str.contains("</answer>", na=False)
)
df = df[mask]
df

,id,image_base64,conversations,instruction,response
0,fab1c569-6e43-48b8-963d-fc68d88b7923,iVBORw0KGgoAAAANSUhEUgAAAXwAAADjCAYAAACGoK/oAA...,"[{'from': 'human', 'value': '<image> Hint: Ple...",Please answer the question and provide the cor...,"<think>\nOkay, so I have this geometry problem..."
1,b64aafe6-8b82-4e71-9c0d-f9b0c02f9dbc,iVBORw0KGgoAAAANSUhEUgAAAhkAAAEMCAYAAABp6aQ+AA...,"[{'from': 'human', 'value': '<image> Hint: Ple...",Please answer the question and provide the cor...,"<think>\nOkay, so I'm trying to solve this geo..."
2,8c5aa8b2-5f9a-4f61-a898-d027a8a1fe3c,iVBORw0KGgoAAAANSUhEUgAAAg8AAADxCAYAAABMIlTiAA...,"[{'from': 'human', 'value': '<image> Hint: Ple...",Please answer the question and provide the cor...,"<think>\nOkay, so I'm trying to figure out how..."
3,7840e5f5-022a-46be-b0fd-9b688032dd54,iVBORw0KGgoAAAANSUhEUgAAAVkAAADMCAYAAADZLtCfAA...,"[{'from': 'human', 'value': '<image> Hint: Ple...",Please answer the question and provide the cor...,<think>\nThe image shows a rectangle divided i...
4,8656a4ff-1fd0-47ed-9b36-b8131cc0de6f,iVBORw0KGgoAAAANSUhEUgAAAZsAAAEACAYAAABlOdt4AA...,"[{'from': 'human', 'value': '<image> Hint: Ple...",Please answer the question and provide the cor...,"<think>\nOkay, so I'm trying to solve this geo..."
...,...,...,...,...,...
11182,ab5e1b8b-7ae3-46c2-8022-b7ed266fcae0,iVBORw0KGgoAAAANSUhEUgAAAUcAAACICAIAAAD74r7pAA...,"[{'from': 'human', 'value': '<image> Hint: Ple...",Please answer the question and provide the cor...,<think>\nThe image shows a grid with two rows ...
11205,3f90fa06-3b7d-4824-a454-c1abcc5882aa,iVBORw0KGgoAAAANSUhEUgAAAGAAAABgCAIAAABt+uBvAA...,"[{'from': 'human', 'value': '<image> Hint: Ple...",Please answer the question and provide the cor...,"<think>\nOkay, so I need to figure out which c..."
11210,9586239b-278d-45d2-bf07-5a4da90ddbbb,iVBORw0KGgoAAAANSUhEUgAAAJQAAABNCAIAAADPb9pkAA...,"[{'from': 'human', 'value': '<image> Hint: Ple...",Please answer the question and provide the cor...,<think>\nThe image shows a digital clock displ...
11211,12de3e65-acd9-42c1-8573-7dd052ae0b5e,iVBORw0KGgoAAAANSUhEUgAAAU4AAACPCAIAAAAa8qWbAA...,"[{'from': 'human', 'value': '<image> Hint: Ple...",Please answer the question and provide the cor...,"<think>\nOkay, so I have this question about f..."


In [37]:
# Remove duplicates based on instruction text
df = df.drop_duplicates('instruction')
df

,id,image_base64,conversations,instruction,response
0,fab1c569-6e43-48b8-963d-fc68d88b7923,iVBORw0KGgoAAAANSUhEUgAAAXwAAADjCAYAAACGoK/oAA...,"[{'from': 'human', 'value': '<image> Hint: Ple...",Please answer the question and provide the cor...,"<think>\nOkay, so I have this geometry problem..."
1,b64aafe6-8b82-4e71-9c0d-f9b0c02f9dbc,iVBORw0KGgoAAAANSUhEUgAAAhkAAAEMCAYAAABp6aQ+AA...,"[{'from': 'human', 'value': '<image> Hint: Ple...",Please answer the question and provide the cor...,"<think>\nOkay, so I'm trying to solve this geo..."
2,8c5aa8b2-5f9a-4f61-a898-d027a8a1fe3c,iVBORw0KGgoAAAANSUhEUgAAAg8AAADxCAYAAABMIlTiAA...,"[{'from': 'human', 'value': '<image> Hint: Ple...",Please answer the question and provide the cor...,"<think>\nOkay, so I'm trying to figure out how..."
3,7840e5f5-022a-46be-b0fd-9b688032dd54,iVBORw0KGgoAAAANSUhEUgAAAVkAAADMCAYAAADZLtCfAA...,"[{'from': 'human', 'value': '<image> Hint: Ple...",Please answer the question and provide the cor...,<think>\nThe image shows a rectangle divided i...
4,8656a4ff-1fd0-47ed-9b36-b8131cc0de6f,iVBORw0KGgoAAAANSUhEUgAAAZsAAAEACAYAAABlOdt4AA...,"[{'from': 'human', 'value': '<image> Hint: Ple...",Please answer the question and provide the cor...,"<think>\nOkay, so I'm trying to solve this geo..."
...,...,...,...,...,...
11172,e8592bc1-49cf-4172-beb0-a85a8e0a895a,iVBORw0KGgoAAAANSUhEUgAAAjkAAABQCAIAAAB07fOBAA...,"[{'from': 'human', 'value': '<image> Hint: Ple...",Please answer the question and provide the cor...,"<think>\nOkay, so I'm trying to determine whic..."
11182,ab5e1b8b-7ae3-46c2-8022-b7ed266fcae0,iVBORw0KGgoAAAANSUhEUgAAAUcAAACICAIAAAD74r7pAA...,"[{'from': 'human', 'value': '<image> Hint: Ple...",Please answer the question and provide the cor...,<think>\nThe image shows a grid with two rows ...
11205,3f90fa06-3b7d-4824-a454-c1abcc5882aa,iVBORw0KGgoAAAANSUhEUgAAAGAAAABgCAIAAABt+uBvAA...,"[{'from': 'human', 'value': '<image> Hint: Ple...",Please answer the question and provide the cor...,"<think>\nOkay, so I need to figure out which c..."
11210,9586239b-278d-45d2-bf07-5a4da90ddbbb,iVBORw0KGgoAAAANSUhEUgAAAJQAAABNCAIAAADPb9pkAA...,"[{'from': 'human', 'value': '<image> Hint: Ple...",Please answer the question and provide the cor...,<think>\nThe image shows a digital clock displ...


In [ ]:
# Convert base64 strings to PIL images
import base64
from PIL import Image
import io

def base64_to_pil(b64_string):
    if ',' in b64_string:
        b64_string = b64_string.split(',')[1]
    
    img_data = base64.b64decode(b64_string)
    img = Image.open(io.BytesIO(img_data))
    return img

df = df.copy()
df['image'] = df['image_base64'].apply(base64_to_pil)
df = df[['id', 'image', 'instruction', 'response']]

In [ ]:
# Split into train, val, and test sets
from sklearn.model_selection import train_test_split
from datasets import DatasetDict

to_split_df = df.sample(n=DATA_SIZE, random_state=SEED)
train_df, val_test_df = train_test_split(
    to_split_df,
    test_size=TEST_RATIO*2,
    random_state=SEED,
)
val_df, test_df = train_test_split(
    val_test_df,
    test_size=0.5,
    random_state=SEED,
)

print("Train set size:", len(train_df))
print("Validation set size:", len(val_df))
print("Test set size:", len(test_df))

Train set size: 1000
Validation set size: 125
Test set size: 125


In [ ]:
# Convert to Hugging Face Datasets
from datasets import Dataset

train_set = Dataset.from_dict(train_df.to_dict(orient='list'))
val_set = Dataset.from_dict(val_df.to_dict(orient='list'))
test_set = Dataset.from_dict(test_df.to_dict(orient='list'))

print("Train set:")
print(train_set)
print()
print("Validation set:")
print(val_set)
print()
print("Test set:")
print(test_set)

Train set:
Dataset({
    features: ['id', 'image', 'instruction', 'response'],
    num_rows: 1000
})

Validation set:
Dataset({
    features: ['id', 'image', 'instruction', 'response'],
    num_rows: 125
})

Test set:
Dataset({
    features: ['id', 'image', 'instruction', 'response'],
    num_rows: 125
})


In [45]:
# Create a DatasetDict for the splits
dataset_split = DatasetDict({
    'train': train_set,
    'validation': val_set,
    'test': test_set,
})
print(dataset_split)

DatasetDict({
    train: Dataset({
        features: ['id', 'image', 'instruction', 'response'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['id', 'image', 'instruction', 'response'],
        num_rows: 125
    })
    test: Dataset({
        features: ['id', 'image', 'instruction', 'response'],
        num_rows: 125
    })
})


In [52]:
# Push the dataset to Hugging Face Hub
dataset_split.push_to_hub(f'alxxtexxr/{DATA_ID.split('/')[-1]}-{DATA_SIZE/1000}K')

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/10 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  91%|#########1| 1.60MB / 1.75MB            

CommitInfo(commit_url='https://huggingface.co/datasets/alxxtexxr/R1-Onevision-1.25K/commit/720d797a3f11cf0f52b5be334cbaff658decdfed', commit_message='Upload dataset', commit_description='', oid='720d797a3f11cf0f52b5be334cbaff658decdfed', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alxxtexxr/R1-Onevision-1.25K', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alxxtexxr/R1-Onevision-1.25K'), pr_revision=None, pr_num=None)